# QUERY EXPANSION

In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

c:\AGO851_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from groq import Groq

In [4]:
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [5]:
client=Groq()

In [6]:
client

In [7]:
model_name = 'openai/gpt-oss-120b'

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_15804\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [9]:
pdf_folder_location = "tesla-annual-reports"

In [10]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [11]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [12]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [13]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [14]:
len(tesla_10k_chunks)

3337

In [15]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [16]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [17]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [18]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [19]:
chromadb_client.heartbeat()

1780488799477758300

In [20]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
i = 0

while i < len(tesla_10k_chunks):
    vectorstore.add_documents( 
        documents=tesla_10k_chunks[i:i+500], 
        ids=["text_" + str(i) for i in range(i, i+500)] 
    )

    i += 500 
    time.sleep(5)

In [21]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [22]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [23]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [24]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 3
    }
)

In [25]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [26]:
user_input = "Any specific fines levied on the company in 2022?"

In [ ]:
# model_name='openai/gpt-oss-120b'

In [28]:
prompt=[
    {'role':'system','content':query_expansion_system_message},
    {'role':'user','content':user_message_template.format(
        question=user_input
    )}
]

In [45]:
query_expansions=client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=prompt,
    temperature=0
)

In [46]:
print(query_expansions.choices[0].message.content)

What are the specific fines levied on the company in 2022 according to the 10-K report?
What fines did the company incur in 2022 as disclosed in the annual report?
Are there any notable fines or penalties imposed on the company in 2022 as stated in the 10-K filing?


In [47]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [48]:
query_expansions_list

['What are the specific fines levied on the company in 2022 according to the 10-K report?',
 'What fines did the company incur in 2022 as disclosed in the annual report?',
 'Are there any notable fines or penalties imposed on the company in 2022 as stated in the 10-K filing?']

In [49]:
expanded_context_list = []

In [50]:
#d means documnet simple 
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [51]:
len(expanded_context_list)

9

In [52]:
expanded_context_list

['222\n\t\nDecreases\tin\tbalances\trelated\tto\texpiration\tof\tthe\tstatute\tof\tlimitations\n(\n7\n)\nDecember\t31,\t2022\n870\n\t\nIncreases\tin\tbalances\trelated\tto\tprior\tyear\ttax\tpositions\n59\n\t\nDecreases\trelated\tto\tsettlement\twith\ttax\tauthorities\n(\n6\n)\nIncreases\tin\tbalances\trelated\tto\tcurrent\tyear\ttax\tpositions\n255\n\t\nDecreases\tin\tbalances\trelated\tto\texpiration\tof\tthe\tstatute\tof\tlimitations\n(\n4\n)\nDecember\t31,\t2023\n$\n1,174\n\t\nWe\tinclude\tinterest\tand\tpenalties\trelated\tto\tunrecognized\ttax\tbenefits\tin\tincome\ttax\texpense.\tWe\trecognized\tnet\tinterest\tand\tpenalties\trelated\tto\nunrecognized\ttax\tbenefits\tin\tprovision\tfor\tincome\ttaxes\tline\tof\tour\tconsolidated\tstatements\tof\toperations\tof\t$\n17\n\tmillion,\t$\n27\n\tmillion\tand\t$\n4\n\tmillion\tfor\tthe\nyears\tended\tDecember\t31,\t2023,\t2022\tand\t2021,\trespectively.\tAs\tof\tDecember\t31,\t2023,\tand\t2022,\twe\thave\taccrued\t$\n47\n\tmillion\tand\

In [53]:
final_context_documents = set(expanded_context_list)

In [54]:
len(final_context_documents)

6

In [55]:
final_context_documents

{'106\n)\nBuy-outs\tof\tnoncontrolling\tinterests\n(\n15\n)\n—\n—\t\n5\n\t\n—\t\n—\t\n5\n\t\n—\t\n5\n\t\nNet\tincome\n43\n\t\n—\n—\t\n—\t\n—\t\n5,519\n\t\n5,519\n\t\n82\n\t\n5,601\n\t\nOther\tcomprehensive\tloss\n—\t\n—\n—\t\n—\t\n(\n309\n)\n—\t\n(\n309\n)\n—\t\n(\n309\n)\nBalance\tas\tof\tDecember\t31,\t2021\n$\n568\n\t\n3,100\n$\n3\n\t\n$\n29,803\n\t\n$\n54\n\t\n$\n329\n\t\n$\n30,189\n\t\n$\n826\n\t\n$\n31,015\n\t\nSettlements\tof\twarrants\n—\t\n37\n—\t\n—\t\n—\t\n—\t\n—\t\n—\t\n—\n\t\nIssuance\tof\tcommon\tstock\tfor\tequity\tincentive\tawards\n—\t\n27\n—\t\n541\n\t\n—\t\n—\t\n541\n\t\n—\t\n541\n\t\nStock-based\tcompensation\n—\t\n—\n—\t\n1,806\n\t\n—\t\n—\t\n1,806\n\t\n—\t\n1,806\n\t\nDistributions\tto\tnoncontrolling\tinterests\n(\n46\n)\n—\n—\t\n—\t\n—\t\n—\t\n—\t\n(\n113\n)\n(\n113\n)\nBuy-outs\tof\tnoncontrolling\tinterests\n(\n11\n)\n—\n—\t\n27\n\t\n—\t\n—\t\n27\n\t\n(\n61\n)\n(\n34\n)\nNet\t(loss)\tincome\n(\n102\n)\n—\n—\t\n—\t\n—\t\n12,556\n\t\n12,556\n\t\n133\n\t\n12,689\

In [56]:
system_message_query_expansion="""
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [57]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [58]:
final_prompt=[
    {
        'role':'system', 'content':system_message_query_expansion
    },
    {
        'role':'user', 'content':qna_user_message_template.format(
            context=final_context_documents,
            question=user_input
        )
    }
]

In [59]:
response=client.chat.completions.create(
    model=model_name,
    messages=final_prompt,
    temperature=0
)

In [60]:
print(response.choices[0].message.content)

The company recorded $27 million of interest and penalties related to unrecognized tax benefits for the year ended December 31 2022.
